# LeetCode 1791 — Find Center of Star Graph bằng Python

Link đề: https://leetcode.com/problems/find-center-of-star-graph/

Độ khó: **Easy** · Chủ đề: Graph

```python
class Solution:
    def findCenter(self, edges: List[List[int]]) -> int:

```

**Mục tiêu học:**

1. Làm quen từ vựng **đồ thị (graph)**: đỉnh (node), cạnh (edge), vô hướng (undirected), **bậc (degree)**.
2. Biết biểu diễn đồ thị bằng **danh sách cạnh** và **danh sách kề (adjacency list)** trong Python.
3. Giải bằng cách đếm bậc `O(n)` — kỹ thuật nền tảng cho mọi bài đồ thị.
4. Rút ra insight: tâm là đỉnh **chung của hai cạnh bất kỳ** → lời giải `O(1)`.
5. Biết dùng `dict`, `defaultdict`, `set` và phép giao `&`.

## 1. Bài toán yêu cầu gì?

Có một **đồ thị hình sao (star graph)** vô hướng gồm `n` đỉnh đánh số `1..n`. Đồ thị hình sao có **một đỉnh tâm** và đúng `n - 1` cạnh, mỗi cạnh nối tâm với một đỉnh khác.

Cho `edges`, trong đó `edges[i] = [u, v]` là một cạnh giữa `u` và `v`. Trả về **đỉnh tâm**.

| `edges` | Tâm |
|---|:---:|
| `[[1,2],[2,3],[4,2]]` | `2` |
| `[[1,2],[5,1],[1,3],[1,4]]` | `1` |

**Ràng buộc:** `3 <= n <= 10^5`, `len(edges) == n - 1`, `edges[i].length == 2`, `1 <= u, v <= n`, `u != v`, dữ liệu **luôn** là một đồ thị hình sao hợp lệ.

Hình dung ví dụ 1:

```text
    1
    |
3 — 2 — 4
```

In [1]:
from typing import List

examples = [
    ([[1, 2], [2, 3], [4, 2]], 2),
    ([[1, 2], [5, 1], [1, 3], [1, 4]], 1),
]
for edges, expected in examples:
    print("edges =", edges, "-> tâm kỳ vọng", expected)

edges = [[1, 2], [2, 3], [4, 2]] -> tâm kỳ vọng 2
edges = [[1, 2], [5, 1], [1, 3], [1, 4]] -> tâm kỳ vọng 1


`from typing import List` cho phép viết gợi ý kiểu `List[List[int]]` như chữ ký LeetCode. Nó **không** ảnh hưởng tới việc chạy code — chỉ để đọc dễ hơn.

## 2. Nền tảng: đồ thị là gì?

### 2.1. Đỉnh, cạnh, vô hướng

- **Đỉnh (node/vertex):** một "điểm", ở đây là số `1..n`.
- **Cạnh (edge):** một đường nối hai đỉnh. `[1, 2]` nghĩa là 1 nối với 2.
- **Vô hướng:** `[1, 2]` và `[2, 1]` là **cùng một** cạnh — đi được cả hai chiều.

### 2.2. Bậc (degree)

**Bậc** của một đỉnh = số cạnh chạm vào đỉnh đó.

Trong đồ thị hình sao `n` đỉnh: tâm có bậc `n - 1`, mọi đỉnh khác có bậc `1`.

In [2]:
edges = [[1, 2], [2, 3], [4, 2]]
n = len(edges) + 1            # star graph: số cạnh = n - 1
print("số đỉnh n =", n)

# Đếm bậc bằng tay cho từng đỉnh
for node in range(1, n + 1):
    deg = sum(1 for u, v in edges if u == node or v == node)
    print(f"đỉnh {node}: bậc {deg}")

số đỉnh n = 4
đỉnh 1: bậc 1
đỉnh 2: bậc 3
đỉnh 3: bậc 1
đỉnh 4: bậc 1


### 2.3. Unpacking trong vòng lặp: `for u, v in edges`

Mỗi phần tử của `edges` là một list 2 phần tử. Viết `for u, v in edges` sẽ tự tách thành `u` và `v`.

In [3]:
for u, v in [[1, 2], [2, 3], [4, 2]]:
    print("u =", u, "| v =", v)

u = 1 | v = 2
u = 2 | v = 3
u = 4 | v = 2


### 2.4. `dict` để đếm, `dict.get(key, 0)`

`deg = {}` là một từ điển rỗng. `deg.get(x, 0)` trả về giá trị của `x` nếu có, không có thì trả `0` → tiện để cộng dồn.

In [4]:
deg = {}
for x in [1, 2, 2, 3, 2]:
    deg[x] = deg.get(x, 0) + 1
print(deg)                         # {1: 1, 2: 3, 3: 1}
print("đỉnh có bậc lớn nhất:", max(deg, key=deg.get))

{1: 1, 2: 3, 3: 1}
đỉnh có bậc lớn nhất: 2


`max(deg, key=deg.get)` duyệt các **key** của `deg` và chọn key có `deg.get(key)` lớn nhất.

### 2.5. Danh sách kề với `defaultdict(list)`

**Adjacency list**: với mỗi đỉnh, lưu list các đỉnh kề nó. `defaultdict(list)` tự tạo list rỗng khi gặp key mới, không cần kiểm tra `if key in ...`.

In [5]:
from collections import defaultdict

adj = defaultdict(list)
for u, v in [[1, 2], [2, 3], [4, 2]]:
    adj[u].append(v)
    adj[v].append(u)       # vô hướng: thêm cả hai chiều
print(dict(adj))
for node, neighbors in adj.items():
    print(f"đỉnh {node} kề {neighbors} -> bậc {len(neighbors)}")

{1: [2], 2: [1, 3, 4], 3: [2], 4: [2]}
đỉnh 1 kề [2] -> bậc 1
đỉnh 2 kề [1, 3, 4] -> bậc 3
đỉnh 3 kề [2] -> bậc 1
đỉnh 4 kề [2] -> bậc 1


### 2.6. `set` và phép giao `&`

`set` là tập hợp không trùng lặp. `a & b` cho các phần tử **chung**.

In [6]:
a = {1, 2}
b = {2, 3}
print("a & b =", a & b)            # {2}
print("lấy phần tử duy nhất:", (a & b).pop())

a & b = {2}
lấy phần tử duy nhất: 2


## 3. Cách trực giác — đếm bậc mọi đỉnh

Tâm là đỉnh có bậc `n - 1` (cũng là đỉnh duy nhất có bậc > 1). Vậy:

1. Duyệt mọi cạnh `[u, v]`, cộng bậc cho `u` và `v`.
2. Trả về đỉnh có bậc lớn nhất.

In [7]:
def find_center_degree(edges: List[List[int]]) -> int:
    deg = {}
    for u, v in edges:
        deg[u] = deg.get(u, 0) + 1
        deg[v] = deg.get(v, 0) + 1
    return max(deg, key=deg.get)

for edges, expected in examples:
    print(find_center_degree(edges), "| kỳ vọng", expected)

2 | kỳ vọng 2
1 | kỳ vọng 1


### 3.1. Biến thể của baseline: dùng list thay dict

Vì đỉnh là `1..n`, ta có thể dùng list `deg` độ dài `n + 1` (bỏ chỉ số 0). Dừng ngay khi một đỉnh đạt bậc `2` — chỉ tâm mới có bậc ≥ 2.

In [8]:
def find_center_degree_list(edges: List[List[int]]) -> int:
    n = len(edges) + 1
    deg = [0] * (n + 1)
    for u, v in edges:
        deg[u] += 1
        deg[v] += 1
        if deg[u] == 2:
            return u
        if deg[v] == 2:
            return v
    return -1   # không xảy ra với input hợp lệ

for edges, expected in examples:
    print(find_center_degree_list(edges), "| kỳ vọng", expected)

2 | kỳ vọng 2
1 | kỳ vọng 1


## 4. Nút thắt và insight

Baseline duyệt **tất cả** `n - 1` cạnh và tốn bộ nhớ `O(n)`. Nhưng hãy nhìn kỹ cấu trúc sao:

- **Mọi** cạnh đều chứa tâm.
- Hai cạnh khác nhau `[c, x]` và `[c, y]` có `x != y` (không có cạnh lặp), nên **đỉnh chung duy nhất** của chúng là `c`.

> **Insight:** Chỉ cần nhìn **hai cạnh đầu tiên**. Đỉnh xuất hiện trong cả hai chính là tâm.

Vì `n >= 3` nên luôn có ít nhất 2 cạnh.

In [9]:
for edges, expected in examples:
    e0, e1 = edges[0], edges[1]
    print("cạnh 0:", e0, "| cạnh 1:", e1, "| đỉnh chung:", set(e0) & set(e1))

cạnh 0: [1, 2] | cạnh 1: [2, 3] | đỉnh chung: {2}
cạnh 0: [1, 2] | cạnh 1: [5, 1] | đỉnh chung: {1}


## 5. Lời giải chính — so sánh hai cạnh đầu

Gọi `edges[0] = [a, b]`, `edges[1] = [c, d]`. Tâm là `a` hoặc `b` (vì tâm thuộc mọi cạnh).

- Nếu `a == c` hoặc `a == d` → `a` thuộc cả hai cạnh → `a` là tâm.
- Ngược lại → tâm phải là `b`.

Chỉ cần 2 phép so sánh!

In [10]:
# Từng đoạn: lấy 4 đỉnh của hai cạnh đầu
edges = [[1, 2], [2, 3], [4, 2]]
a, b = edges[0]
c, d = edges[1]
print("a, b =", a, b, "| c, d =", c, d)
print("a == c hoặc a == d?", a == c or a == d)

a, b = 1 2 | c, d = 2 3
a == c hoặc a == d? False


In [11]:
def find_center(edges: List[List[int]]) -> int:
    a, b = edges[0]
    c, d = edges[1]
    if a == c or a == d:
        return a
    return b

for edges, expected in examples:
    print(find_center(edges), "| kỳ vọng", expected)

2 | kỳ vọng 2
1 | kỳ vọng 1


Mẹo đọc code: `a, b = edges[0]` cũng là **unpacking** — list `[1, 2]` được tách thành `a = 1`, `b = 2`. Trong code nộp LeetCode ta đổi tên cạnh thứ hai thành `x, y` cho dễ phân biệt với cạnh đầu.

## 6. TRACE từng bước

Trace ví dụ 2: `edges = [[1,2],[5,1],[1,3],[1,4]]`

| Bước | Biến | Giá trị | Ý nghĩa |
|---|---|---|---|
| 1 | `a, b` | `1, 2` | tâm ∈ {1, 2} |
| 2 | `c, d` | `5, 1` | tâm ∈ {5, 1} |
| 3 | `a == c`? | `1 == 5` → False | |
| 4 | `a == d`? | `1 == 1` → True | `1` thuộc cả hai cạnh |
| 5 | return | `1` | |

Không cần nhìn các cạnh `[1,3]`, `[1,4]`.

In [12]:
def trace_find_center(edges: List[List[int]]) -> int:
    print("edges =", edges)
    a, b = edges[0]
    print(f"cạnh 0: a={a}, b={b} -> tâm nằm trong {{{a}, {b}}}")
    x, y = edges[1]
    print(f"cạnh 1: x={x}, y={y} -> tâm nằm trong {{{x}, {y}}}")
    if a == x or a == y:
        print(f"a={a} có mặt trong cạnh 1 -> tâm = {a}")
        return a
    print(f"a={a} không có trong cạnh 1 -> tâm = b = {b}")
    return b

trace_find_center([[1, 2], [5, 1], [1, 3], [1, 4]])
print()
trace_find_center([[1, 2], [2, 3], [4, 2]])

edges = [[1, 2], [5, 1], [1, 3], [1, 4]]
cạnh 0: a=1, b=2 -> tâm nằm trong {1, 2}
cạnh 1: x=5, y=1 -> tâm nằm trong {5, 1}
a=1 có mặt trong cạnh 1 -> tâm = 1

edges = [[1, 2], [2, 3], [4, 2]]
cạnh 0: a=1, b=2 -> tâm nằm trong {1, 2}
cạnh 1: x=2, y=3 -> tâm nằm trong {2, 3}
a=1 không có trong cạnh 1 -> tâm = b = 2


2

### 6.1. Trace baseline đếm bậc (để thấy nó làm nhiều việc hơn)

In [13]:
def trace_degree(edges: List[List[int]]) -> int:
    deg = {}
    for step, (u, v) in enumerate(edges, start=1):
        deg[u] = deg.get(u, 0) + 1
        deg[v] = deg.get(v, 0) + 1
        print(f"bước {step}: cạnh [{u},{v}] -> deg = {deg}")
    center = max(deg, key=deg.get)
    print("đỉnh bậc lớn nhất:", center, "với bậc", deg[center])
    return center

trace_degree([[1, 2], [5, 1], [1, 3], [1, 4]])

bước 1: cạnh [1,2] -> deg = {1: 1, 2: 1}
bước 2: cạnh [5,1] -> deg = {1: 2, 2: 1, 5: 1}
bước 3: cạnh [1,3] -> deg = {1: 3, 2: 1, 5: 1, 3: 1}
bước 4: cạnh [1,4] -> deg = {1: 4, 2: 1, 5: 1, 3: 1, 4: 1}
đỉnh bậc lớn nhất: 1 với bậc 4


1

`enumerate(edges, start=1)` đánh số bắt đầu từ 1 thay vì 0 — tiện cho in "bước 1, bước 2…".

## 7. Các lời giải khác

### Cách 3 — Giao hai tập hợp

```python
return (set(edges[0]) & set(edges[1])).pop()
```

Ngắn, rõ ý "đỉnh chung", nhưng tạo 2 set nhỏ (vẫn `O(1)`).

### Cách 4 — Danh sách kề, tìm đỉnh có `n - 1` hàng xóm

Tổng quát nhất (dùng được khi cần thông tin khác của đồ thị), nhưng tốn `O(n)` bộ nhớ.

In [14]:
def find_center_set(edges: List[List[int]]) -> int:
    return (set(edges[0]) & set(edges[1])).pop()

def find_center_adj(edges: List[List[int]]) -> int:
    n = len(edges) + 1
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    for node, nbrs in adj.items():
        if len(nbrs) == n - 1:
            return node
    return -1

for edges, expected in examples:
    print(find_center_set(edges), find_center_adj(edges), "| kỳ vọng", expected)

2 2 | kỳ vọng 2
1 1 | kỳ vọng 1


### 7.1. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Đếm bậc (dict) | đỉnh bậc lớn nhất | `O(n)` | `O(n)` | tổng quát, dễ nghĩ |
| 2. Đếm bậc (list, dừng sớm) | bậc = 2 là tâm | `O(n)` worst, thường rất sớm | `O(n)` | |
| 3. Giao set | đỉnh chung 2 cạnh | `O(1)` | `O(1)` | ngắn gọn |
| 4. Adjacency list | `len(adj[x]) == n-1` | `O(n)` | `O(n)` | nền tảng cho bài graph khác |
| **5. So sánh 2 cạnh (chính)** | `a in edges[1]`? | `O(1)` | `O(1)` | tối ưu nhất |

`n` = số đỉnh.

Thật ra cách 2 với dữ liệu hợp lệ luôn dừng ở cạnh thứ hai (tâm có mặt ở cạnh 1 và 2) → cũng `O(1)` thời gian, nhưng phải cấp phát list `O(n)`.

## 8. Code nộp LeetCode

In [15]:
class Solution:
    def findCenter(self, edges: List[List[int]]) -> int:
        a, b = edges[0]
        x, y = edges[1]
        return a if a == x or a == y else b

sol = Solution()
for edges, expected in examples:
    print(sol.findCenter(edges), "| kỳ vọng", expected)

2 | kỳ vọng 2
1 | kỳ vọng 1


`A if điều_kiện else B` là **biểu thức điều kiện** (ternary): trả `A` nếu điều kiện đúng, ngược lại `B`. Viết gọn cho `if/else` hai nhánh.

## 9. Vì sao thuật toán đúng?

Gọi tâm là `C`. Theo định nghĩa đồ thị hình sao:

1. **Mọi cạnh chứa `C`** (có `n-1` cạnh, nối `C` với `n-1` đỉnh còn lại).
2. Do đó `edges[0] = [C, p]` (thứ tự bất kỳ) và `edges[1] = [C, q]`, với `p != q` (hai cạnh khác nhau tới hai lá khác nhau).
3. `edges[0] ∩ edges[1] = {C}` vì `p ∉ edges[1]` (nếu `p ∈ edges[1]` thì `p = q` hoặc `p = C`, đều vô lý).

Code: nếu `a` thuộc `edges[1]` thì `a ∈ edges[0] ∩ edges[1] = {C}` nên `a = C`. Ngược lại `a != C`, mà `C ∈ {a, b}` nên `b = C`.

## 10. Độ phức tạp

- **Lời giải chính:** Time `O(1)` (chỉ đọc 2 cạnh), Space `O(1)`.
- **Đếm bậc:** Time `O(n)`, Space `O(n)` với `n` là số đỉnh.

Đây là ví dụ đẹp cho thấy **cấu trúc đặc biệt của input** (đảm bảo là star graph) cho phép bỏ qua phần lớn dữ liệu.

## 11. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.findCenter([[1, 2], [2, 3], [4, 2]]), 2)
check("đề 2", sol.findCenter([[1, 2], [5, 1], [1, 3], [1, 4]]), 1)
check("n = 3, tâm ở vị trí sau", sol.findCenter([[1, 3], [2, 3]]), 3)
check("n = 3, tâm ở vị trí đầu", sol.findCenter([[3, 1], [3, 2]]), 3)
check("tâm là b của cạnh 0", sol.findCenter([[4, 7], [7, 1], [2, 7]]), 7)
check("tâm là a, nằm ở y", sol.findCenter([[5, 1], [2, 5]]), 5)

PASS đề 1: 2
PASS đề 2: 1
PASS n = 3, tâm ở vị trí sau: 3
PASS n = 3, tâm ở vị trí đầu: 3
PASS tâm là b của cạnh 0: 7
PASS tâm là a, nằm ở y: 5


In [17]:
import random
random.seed(1791)

def random_star(n):
    center = random.randint(1, n)
    others = [v for v in range(1, n + 1) if v != center]
    random.shuffle(others)
    edges = []
    for v in others:
        edges.append([center, v] if random.random() < 0.5 else [v, center])
    return edges, center

solvers = [find_center_degree, find_center_degree_list, find_center, find_center_set,
           find_center_adj, Solution().findCenter]
for _ in range(2000):
    n = random.randint(3, 30)
    edges, center = random_star(n)
    for f in solvers:
        assert f(edges) == center, (f.__name__, edges, center)
print("6 cách đều đúng trên 2000 đồ thị sao ngẫu nhiên (seed 1791)")

6 cách đều đúng trên 2000 đồ thị sao ngẫu nhiên (seed 1791)


In [18]:
# Test hiệu năng: n = 10^5
import time
edges, center = random_star(100_000)
for f in [find_center_degree, find_center]:
    t0 = time.perf_counter()
    r = f(edges)
    dt = time.perf_counter() - t0
    assert r == center
    print(f"{f.__name__:22} đúng, thời gian {'nhanh' if dt < 1 else 'chậm'} (đo được ~{dt*1000:.3f} ms)")

find_center_degree     đúng, thời gian nhanh (đo được ~24.229 ms)
find_center            đúng, thời gian nhanh (đo được ~0.001 ms)


(Số ms đo được thay đổi theo máy; điều quan trọng là cách `O(1)` không phụ thuộc `n`.)

## 12. Lỗi người mới hay gặp

### Lỗi 1: Trả về đỉnh đầu tiên của cạnh đầu

```python
return edges[0][0]
```

Sai khi tâm nằm ở vị trí thứ hai, ví dụ `[[1,2],[2,3]]`.

In [19]:
def wrong_first(edges):
    return edges[0][0]

print("sai:", wrong_first([[1, 2], [2, 3], [4, 2]]), "| đúng:", find_center([[1, 2], [2, 3], [4, 2]]))

sai: 1 | đúng: 2


### Lỗi 2: Chỉ so sánh `a == x` (quên `a == y`)

Vì vô hướng, tâm có thể ở bất kỳ vị trí nào trong cạnh thứ hai.

In [20]:
def wrong_one_compare(edges):
    a, b = edges[0]
    x, y = edges[1]
    return a if a == x else b

print("sai:", wrong_one_compare([[1, 2], [5, 1], [1, 3]]), "| đúng:", find_center([[1, 2], [5, 1], [1, 3]]))

sai: 2 | đúng: 1


### Lỗi 3: Quên thêm cạnh hai chiều khi dựng adjacency list

Chỉ `adj[u].append(v)` → bậc của các đỉnh nằm ở vị trí `v` bị đếm thiếu.

### Lỗi 4: Tạo list bậc độ dài `n` thay vì `n + 1`

Đỉnh đánh số `1..n`, nên `deg[n]` sẽ vượt chỉ số nếu `deg = [0] * n` → `IndexError`.

In [21]:
def wrong_adj_one_direction(edges):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
    return max(adj, key=lambda k: len(adj[k]))

print("sai:", wrong_adj_one_direction([[1, 2], [3, 2], [4, 2]]), "| đúng:", find_center([[1, 2], [3, 2], [4, 2]]))

try:
    n = 4
    deg = [0] * n
    deg[4] += 1
except IndexError as e:
    print("IndexError:", e)

sai: 1 | đúng: 2
IndexError: list index out of range


`lambda k: len(adj[k])` là **hàm vô danh** (anonymous function) một dòng: nhận `k`, trả `len(adj[k])`.

## 13. Mẹo nhận dạng pattern

- Đề cho "danh sách cạnh" → nghĩ tới **bậc** và **adjacency list** — hai công cụ đầu tiên của graph.
- Đề **đảm bảo cấu trúc đặc biệt** (sao, cây, đường thẳng…) → tự hỏi: "có cần đọc hết input không?". Thường chỉ một phần nhỏ đủ để suy ra đáp án.
- "Đỉnh nối với mọi đỉnh khác" ⇔ bậc `n - 1`.
- Bài liên quan: 997 Find the Town Judge (bậc vào/ra), 1557 Minimum Number of Vertices to Reach All Nodes, 2374 Node With Highest Edge Score.

## 14. Biến thể

### 14.1. Kiểm tra một danh sách cạnh **có phải** đồ thị hình sao không

Bây giờ không được giả sử input hợp lệ → phải đọc hết các cạnh.

In [22]:
def is_star_graph(n: int, edges: List[List[int]]) -> bool:
    if len(edges) != n - 1:
        return False
    deg = [0] * (n + 1)
    seen = set()
    for u, v in edges:
        key = (min(u, v), max(u, v))
        if u == v or key in seen:
            return False
        seen.add(key)
        deg[u] += 1
        deg[v] += 1
    return deg.count(n - 1) == 1 and deg.count(1) == n - 1

print(is_star_graph(4, [[1, 2], [2, 3], [4, 2]]))   # True
print(is_star_graph(4, [[1, 2], [2, 3], [3, 4]]))   # False (đường thẳng)

True
False


Chú ý: với `n = 3`… vẫn đúng; còn trường hợp `n = 2` thì cả hai đỉnh đều có bậc 1 = `n-1`, ta không xét vì đề gốc có `n >= 3`.

### 14.2. Find the Town Judge (LeetCode 997) — "tâm" theo bậc có hướng

Thẩm phán được mọi người tin (bậc vào `n-1`) và không tin ai (bậc ra `0`).

In [23]:
def find_judge(n: int, trust: List[List[int]]) -> int:
    score = [0] * (n + 1)
    for a, b in trust:      # a tin b
        score[a] -= 1
        score[b] += 1
    for person in range(1, n + 1):
        if score[person] == n - 1:
            return person
    return -1

print(find_judge(3, [[1, 3], [2, 3]]))           # 3
print(find_judge(3, [[1, 3], [2, 3], [3, 1]]))   # -1

3
-1


## 15. Bài tập luyện tập

**Bài 1 (tính tay):** `edges = [[7, 3], [3, 9], [3, 1], [5, 3]]`. Lập bảng bậc từng đỉnh. Tâm là gì? Hàm `find_center` so sánh những gì?

**Bài 2 (đoán output):** `set([4, 7]) & set([7, 1])` là gì? `max({1: 1, 2: 3}, key={1: 1, 2: 3}.get)` là gì?

**Bài 3 (điền TODO):** Hoàn thiện hàm đếm bậc trả về list `deg` độ dài `n + 1`.

In [24]:
def ex3_degrees(n: int, edges: List[List[int]]) -> List[int]:
    # TODO 1: deg = [0] * (n + 1)
    # TODO 2: với mỗi cạnh u, v: tăng deg[u], deg[v]
    # TODO 3: return deg
    pass

# assert ex3_degrees(4, [[1, 2], [2, 3], [4, 2]]) == [0, 1, 3, 1, 1]

**Bài 4 (tự viết):** Viết `ex4_leaves(edges)` trả về **list các lá** (đỉnh bậc 1) của đồ thị sao, sắp xếp tăng dần. Gợi ý: tìm tâm trước.

In [25]:
def ex4_leaves(edges: List[List[int]]) -> List[int]:
    # TODO
    pass

# assert ex4_leaves([[1, 2], [2, 3], [4, 2]]) == [1, 3, 4]
# assert ex4_leaves([[1, 2], [5, 1], [1, 3], [1, 4]]) == [2, 3, 4, 5]

**Bài 5 (biến thể):** Cho `n` và `edges` (đồ thị vô hướng bất kỳ). Trả về list các đỉnh có bậc lớn nhất (tăng dần).

In [26]:
def ex5_max_degree_nodes(n: int, edges: List[List[int]]) -> List[int]:
    # TODO
    pass

# assert ex5_max_degree_nodes(4, [[1, 2], [2, 3], [3, 4]]) == [2, 3]
# assert ex5_max_degree_nodes(4, [[1, 2], [2, 3], [4, 2]]) == [2]

**Bài 6 (mini challenge):** Một đồ thị "hai sao" gồm hai tâm `C1`, `C2` nối với nhau, mỗi lá nối với đúng một trong hai tâm, mỗi tâm có ít nhất một lá. Viết `ex6_two_centers(edges)` trả về `sorted([C1, C2])`. Gợi ý: hai tâm là hai đỉnh có bậc ≥ 2.

In [27]:
def ex6_two_centers(edges: List[List[int]]) -> List[int]:
    # TODO
    pass

# assert ex6_two_centers([[1, 2], [1, 3], [2, 4], [2, 5]]) == [1, 2]
# assert ex6_two_centers([[6, 1], [6, 9], [9, 2], [9, 3], [4, 6]]) == [6, 9]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | đếm số lần xuất hiện | 3 xuất hiện 4 lần | `a=7` không trong `[3,9]` → `b=3` |
| 2 | phần tử chung | key có value lớn nhất | `{7}`, `2` |
| 3 | list đếm theo chỉ số đỉnh | `deg[u] += 1` | nhớ `n + 1` |
| 4 | lá = mọi đỉnh khác tâm | gom `u`/`v` khác tâm | `sorted(...)` |
| 5 | đếm bậc, tìm max | `best = max(deg[1:])` | lọc `deg[i] == best` |
| 6 | bậc ≥ 2 | đếm bậc bằng dict | `sorted(k for k in deg if deg[k] >= 2)` |

## 17. Đáp án tham khảo

**Bài 1:** bậc: 7→1, 3→4, 9→1, 1→1, 5→1. Tâm `3`. `find_center` lấy `a=7, b=3`, `x=3, y=9`; `7` không bằng `3` hay `9` → trả `b = 3`.

**Bài 2:** `{7}` và `2`.

In [28]:
print(trace_find_center([[7, 3], [3, 9], [3, 1], [5, 3]]))
print(set([4, 7]) & set([7, 1]), max({1: 1, 2: 3}, key={1: 1, 2: 3}.get))

def ex3_degrees_ans(n, edges):
    deg = [0] * (n + 1)
    for u, v in edges:
        deg[u] += 1
        deg[v] += 1
    return deg

def ex4_leaves_ans(edges):
    center = find_center(edges)
    return sorted(v if u == center else u for u, v in edges)

def ex5_max_degree_nodes_ans(n, edges):
    deg = ex3_degrees_ans(n, edges)
    best = max(deg[1:])
    return [i for i in range(1, n + 1) if deg[i] == best]

def ex6_two_centers_ans(edges):
    deg = {}
    for u, v in edges:
        deg[u] = deg.get(u, 0) + 1
        deg[v] = deg.get(v, 0) + 1
    return sorted(k for k in deg if deg[k] >= 2)

assert ex3_degrees_ans(4, [[1, 2], [2, 3], [4, 2]]) == [0, 1, 3, 1, 1]
assert ex4_leaves_ans([[1, 2], [2, 3], [4, 2]]) == [1, 3, 4]
assert ex4_leaves_ans([[1, 2], [5, 1], [1, 3], [1, 4]]) == [2, 3, 4, 5]
assert ex5_max_degree_nodes_ans(4, [[1, 2], [2, 3], [3, 4]]) == [2, 3]
assert ex5_max_degree_nodes_ans(4, [[1, 2], [2, 3], [4, 2]]) == [2]
assert ex6_two_centers_ans([[1, 2], [1, 3], [2, 4], [2, 5]]) == [1, 2]
assert ex6_two_centers_ans([[6, 1], [6, 9], [9, 2], [9, 3], [4, 6]]) == [6, 9]
print("Đáp án tham khảo PASS toàn bộ test!")

edges = [[7, 3], [3, 9], [3, 1], [5, 3]]
cạnh 0: a=7, b=3 -> tâm nằm trong {7, 3}
cạnh 1: x=3, y=9 -> tâm nằm trong {3, 9}
a=7 không có trong cạnh 1 -> tâm = b = 3
3
{7} 2
Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findCenter(self, edges: List[List[int]]) -> int:
        a, b = edges[0]
        x, y = edges[1]
        return a if a == x or a == y else b
```

Template đếm bậc (dùng cho bài graph tổng quát):

```python
deg = [0] * (n + 1)
for u, v in edges:
    deg[u] += 1
    deg[v] += 1
```

**Checklist:**

- [ ] Hiểu tâm thuộc **mọi** cạnh → hai cạnh đầu là đủ?
- [ ] So sánh `a` với **cả hai** đầu của cạnh thứ hai?
- [ ] Nếu dựng adjacency list: thêm **hai chiều**?
- [ ] List bậc có độ dài `n + 1` khi đỉnh đánh số từ 1?
- [ ] Tên method `findCenter`, trả `int`?